# 07 — Seed Harness on Full-Concat Scrubbed+ Text

In [1]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    balanced_accuracy_score, f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
    EarlyStoppingCallback, Trainer, TrainingArguments,
)
from transformers.utils.notebook import NotebookProgressCallback

RANDOM_STATE = 42
SEEDS = [42, 123, 2024]
MIN_SAMPLES_PER_CLASS = 100
TEXT_COLUMN = 'text_full_concat_scrubbed_plus'
MAX_LENGTH = 256

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
USE_FP16 = True

ROBERTA_CKPT = 'roberta-base'
ROBERTA_BEST_LR = 2e-5
ROBERTA_BEST_WEIGHTED = True

MODERNBERT_CKPT = 'answerdotai/ModernBERT-base'
MODERNBERT_BEST_LR = 3e-5
MODERNBERT_BEST_WEIGHTED = False

OUTPUT_DIR_ROOT = 'artifacts/seed_harness_fulltext_scrubbed_plus'
os.makedirs(OUTPUT_DIR_ROOT, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


Device: cuda


## Scrubbing vocabulary

In [2]:
# Scrubbing vocabulary: countries + region aliases + cultivars + producer context
# Lifted from 04.5 so that 02.1 / 03.1 / 04.1.1 / 06.1 / 07.1 are a fair comparison.
COUNTRY_TERMS = {
    "Ethiopia": ["ethiopia", "ethiopian"],
    "Colombia": ["colombia", "colombian", "columbian"],
    "Panama": ["panama", "panamanian"],
    "Kenya": ["kenya", "kenyan"],
    "Indonesia": ["indonesia", "indonesian"],
    "Guatemala": ["guatemala", "guatemalan"],
    "Costa Rica": ["costa rica", "costa rican", "costarican"],
    "El Salvador": ["el salvador", "salvadoran", "salvadorean", "salvadorian"],
    "Rwanda": ["rwanda", "rwandan"],
    "Brazil": ["brazil", "brazilian"],
    "Honduras": ["honduras", "honduran"],
    "Peru": ["peru", "peruvian"],
    "Taiwan": ["taiwan", "taiwanese"],
    "Papua New Guinea": ["papua new guinea", "papua new guinean"],
    "Nicaragua": ["nicaragua", "nicaraguan"],
    "Burundi": ["burundi", "burundian"],
    "Thailand": ["thailand", "thai"],
    "India": ["india", "indian"],
    "Mexico": ["mexico", "mexican"],
    "Tanzania": ["tanzania", "tanzanian"],
    "Yemen": ["yemen", "yemeni"],
    "Ecuador": ["ecuador", "ecuadorian"],
    "Bolivia": ["bolivia", "bolivian"],
    "Jamaica": ["jamaica", "jamaican"],
    "Uganda": ["uganda", "ugandan"],
    "Dominican Republic": ["dominican republic", "dominican"],
    "Zambia": ["zambia", "zambian"],
    "China": ["china", "chinese"],
    "Vietnam": ["vietnam", "vietnamese"],
    "Philippines": ["philippines", "philippine", "filipino"],
    "Malaysia": ["malaysia", "malaysian"],
    "Laos": ["laos", "laotian"],
    "Zimbabwe": ["zimbabwe", "zimbabwean"],
    "Haiti": ["haiti", "haitian"],
    "Puerto Rico": ["puerto rico", "puerto rican", "puertorican"],
    "Nepal": ["nepal", "nepalese", "nepali"],
    "Myanmar": ["myanmar", "burmese"],
    "Cameroon": ["cameroon", "cameroonian"],
    "Australia": ["australia", "australian"],
    "South Africa": ["south africa", "south african"],
    "Malawi": ["malawi", "malawian"],
    "Venezuela": ["venezuela", "venezuelan", "merida state", "mocoties valley"],
    "Timor-Leste": ["east timor", "timor leste", "timorese"],
    "DR Congo": ["democratic republic of the congo", "dr congo", "congo", "congolese"],
    "United Kingdom": ["united kingdom", "british", "pitcairn island", "saint helena", "st helena", "sandy bay valley"],
    "United States": [
        "usa", "united states", "american",
        "hawaii", "hawai'i", "hawaiian", "hawaii island",
        "big island", "kona", "puna district", "holualoa", "oahu", "maui", "kauai",
        "ka u", "ka'u", "kau",
    ],
}

REGION_ALIASES = {
    "Ethiopia": ["yirgacheffe", "sidamo", "sidama", "guji", "gedeb", "gedeo", "kochere", "hambela",
                 "shakiso", "jimma", "limu", "oromia", "harrar", "kaffa", "bench maji", "bench-maji",
                 "arbegona", "bensa"],
    "Colombia": ["huila", "cauca", "narino", "tolima", "quindio", "caldas", "risaralda", "antioquia",
                 "cundinamarca", "santander", "pitalito", "acevedo", "planadas", "gaitania", "piendamo",
                 "caicedonia", "san agustin", "armenia"],
    "Panama": ["boquete", "chiriqui", "volcan", "jaramillo", "alto quiel", "paso ancho",
               "piedra candela", "canas verdes", "silla del pando", "renacimiento"],
    "Kenya": ["nyeri", "kirinyaga", "kiambu", "embu", "muranga", "murang'a", "thika", "ruiru",
              "mathira", "meru", "nakuru", "gichugu", "karatina"],
    "Indonesia": ["sumatra", "aceh", "gayo", "lintong", "mandheling", "sidikalang", "toraja",
                  "sulawesi", "java", "bali", "kintamani", "flores", "kerinci"],
    "Guatemala": ["huehuetenango", "antigua", "acatenango", "fraijanes", "coban", "atitlan",
                  "chimaltenango", "quiche", "solola", "palencia", "sacatepequez", "san marcos",
                  "hoja blanca", "cuilco"],
    "Costa Rica": ["tarrazu", "central valley", "west valley", "tres rios", "naranjo", "dota",
                   "poas", "alajuela", "brunca", "turrialba", "coto brus", "chirripo"],
    "El Salvador": ["ahuachapan", "chalatenango", "apaneca", "ilamatepec", "santa ana",
                    "el boqueron", "quetzaltepec", "juayua", "ataco"],
    "Rwanda": ["gakenke", "nyamasheke", "karongi", "huye", "nyamagabe", "rulindo", "gikongoro",
               "lake kivu"],
    "Brazil": ["cerrado", "mogiana", "minas gerais", "mantiqueira", "sul de minas",
               "chapada diamantina", "carmo de minas"],
    "Honduras": ["marcala", "copan", "ocotepeque", "comayagua", "intibuca", "santa barbara",
                 "el paraiso", "capucas"],
    "Peru": ["cajamarca", "jaen", "san ignacio", "chanchamayo", "cusco", "villa rica", "oxapampa",
             "junin", "puno"],
    "Taiwan": ["alishan", "yunlin", "chiayi", "chia yi", "nantou", "taichung", "pingtung"],
    "Papua New Guinea": ["wahgi valley", "western highlands", "eastern highlands", "jiwaka",
                         "kainantu", "okapa"],
    "Nicaragua": ["jinotega", "matagalpa", "nueva segovia", "madriz", "ocotal", "dipilto"],
    "Burundi": ["kayanza", "ngozi", "muramvya", "muyinga", "bururi"],
    "Thailand": ["chiang rai", "doi chang", "doi pangkhon", "chiang mai", "nan province"],
    "India": ["coorg", "chikmagalur", "karnataka", "bababudangiri", "nilgiris"],
    "Mexico": ["chiapas", "oaxaca", "veracruz", "coatepec", "pluma hidalgo"],
    "Tanzania": ["mbeya", "ruvuma", "ngorongoro", "arusha", "kilimanjaro", "mbozi"],
    "Yemen": ["haraaz", "haraz", "sanaa", "bani matar", "hayma"],
    "Ecuador": ["loja", "pichincha", "imbabura", "zamora", "chimborazo", "saraguro"],
    "Bolivia": ["caranavi", "yungas", "la paz"],
    "Jamaica": ["blue mountain", "blue mountains"],
    "Uganda": ["rwenzori", "bugisu", "sipi falls"],
    "Vietnam": ["lam dong", "quang tri", "dalat", "cau dat"],
    "Philippines": ["benguet", "bukidnon", "davao"],
    "China": ["yunnan", "baoshan", "puer", "pu'er", "lincong"],
    "Puerto Rico": ["utuado", "yauco", "adjuntas"],
    "Venezuela": ["mocoties valley", "merida state"],
    "Malawi": ["malawi"],
}

REGION_ONLY_TERMS = [
    "central america", "south america", "latin america",
    "east africa", "central africa", "west africa", "africa",
    "asia", "the americas", "americas",
    "central and south america", "south and central america",
    "east and central africa", "various africa growing regions",
]

CULTIVAR_TERMS = [
    "sl28", "sl-28", "sl 28", "sl34", "sl-34", "sl 34",
    "ruiru 11", "ruiru-11", "batian", "k7",
    "gesha", "geisha",
    "pacamara", "pache", "villa sarchi",
    "bourbon", "pink bourbon", "yellow bourbon", "red bourbon",
    "caturra", "catuai", "catuai vermelho", "catuai amarelo",
    "typica", "mundo novo",
    "maragogipe", "maragogype", "maracaturra",
    "castillo", "colombia variety", "variedad colombia", "tabi",
    "heirloom", "ethiopian heirloom",
    "tim tim", "s795", "ateng",
    "catimor", "sarchimor", "icatu", "obata",
]

PRODUCER_CONTEXT_TERMS = [
    "finca", "hacienda", "beneficio",
    "cup of excellence",
    "washing station", "wet mill",
    "best of panama",
]

all_scrub_terms = []
for v in COUNTRY_TERMS.values():    all_scrub_terms.extend(v)
for v in REGION_ALIASES.values():   all_scrub_terms.extend(v)
all_scrub_terms.extend(REGION_ONLY_TERMS)
all_scrub_terms.extend(CULTIVAR_TERMS)
all_scrub_terms.extend(PRODUCER_CONTEXT_TERMS)
all_scrub_terms = sorted(set(all_scrub_terms), key=len, reverse=True)
print(f'Loaded {len(all_scrub_terms)} scrub terms')

SCRUB_PATTERN = re.compile(
    r'\b(' + '|'.join(re.escape(t) for t in all_scrub_terms) + r')\b',
    flags=re.IGNORECASE,
)

def scrub(text):
    if not isinstance(text, str) or not text:
        return ''
    return SCRUB_PATTERN.sub(' [ORIGIN] ', text)


Loaded 403 scrub terms


## Build text column

In [3]:
# Concat Blind Assessment + Notes + Who Should Drink It + Bottom Line, then scrub
def minimal_raw_text(text):
    text = '' if pd.isna(text) else str(text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def normalize_text_keep_brackets(text):
    text = minimal_raw_text(text)
    text = re.sub(r'[^a-z0-9\s\[\]]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

EXTRA_TEXT_COLS = ['Blind Assessment', 'Notes', 'Who Should Drink It', 'Bottom Line']

def concat_and_scrub(row):
    pieces = []
    for col in EXTRA_TEXT_COLS:
        v = row[col]
        if pd.notna(v) and str(v).strip():
            raw = minimal_raw_text(v)
            scrubbed = scrub(raw)
            cleaned = normalize_text_keep_brackets(scrubbed)
            pieces.append(cleaned)
    return ' '.join(p for p in pieces if p)

df = pd.read_csv('Data/final_coffee_reviews.csv')
df['text_full_concat_scrubbed_plus'] = df.apply(concat_and_scrub, axis=1)
df['text_raw_minimal'] = df['Blind Assessment'].fillna('').map(minimal_raw_text)
df['origin_country'] = df['Country'].astype('string').str.strip().replace('', pd.NA)

work = df[(df['text_raw_minimal'].str.len() >= 30) & (df['origin_country'].notna())].copy()
counts = work['origin_country'].value_counts()
valid_classes = counts[counts >= MIN_SAMPLES_PER_CLASS].index
work = work[work['origin_country'].isin(valid_classes)].copy().reset_index(drop=True)

# Leakage audit
def contains_own_country(row):
    t = row['text_full_concat_scrubbed_plus'].lower()
    c = str(row['origin_country']).lower()
    return c in t if c and t else False
work['leaks_country'] = work.apply(contains_own_country, axis=1)
leak_rate = float(work['leaks_country'].mean())

print(f'Rows: {len(work)} | Classes: {work["origin_country"].nunique()}')
print(f'Avg scrubbed+ text length (chars): {int(work["text_full_concat_scrubbed_plus"].str.len().mean())}')
print(f'Country-name leakage: {leak_rate:.1%}  (target: <2%)')


Rows: 6820 | Classes: 15
Avg scrubbed+ text length (chars): 875
Country-name leakage: 0.8%  (target: <2%)


## Split + labels + class weights

In [4]:
y = work['origin_country']
row_idx = work.index
train_idx, temp_idx, y_train, y_temp = train_test_split(
    row_idx, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y,
)
val_idx, test_idx, y_val, y_test = train_test_split(
    temp_idx, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp,
)

label_names = sorted(work['origin_country'].unique())
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for label, idx in label2id.items()}

train_texts = work.loc[train_idx, TEXT_COLUMN].fillna('').tolist()
val_texts   = work.loc[val_idx,   TEXT_COLUMN].fillna('').tolist()
test_texts  = work.loc[test_idx,  TEXT_COLUMN].fillna('').tolist()
train_labels = work.loc[train_idx, 'origin_country'].map(label2id).tolist()
val_labels   = work.loc[val_idx,   'origin_country'].map(label2id).tolist()
test_labels  = work.loc[test_idx,  'origin_country'].map(label2id).tolist()

class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(label_names)),
    y=np.array(train_labels),
)
class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float)
print('Train/Val/Test:', len(train_idx), len(val_idx), len(test_idx))


Train/Val/Test: 4774 1023 1023


## Dataset + metrics + run_one

In [5]:
class CoffeeOriginDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(texts, truncation=True, max_length=max_length, padding=False)
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {
        'accuracy': accuracy_score(labels, preds),
        'balanced_accuracy': balanced_accuracy_score(labels, preds),
        'precision_macro': precision, 'recall_macro': recall, 'f1_macro': f1,
    }

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def run_one(model_checkpoint, learning_rate, use_class_weights, seed, epochs, early_stop_patience, tag):
    set_seed(seed)
    out_dir = os.path.join(OUTPUT_DIR_ROOT, tag)

    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    train_ds = CoffeeOriginDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
    val_ds   = CoffeeOriginDataset(val_texts,   val_labels,   tokenizer, MAX_LENGTH)
    test_ds  = CoffeeOriginDataset(test_texts,  test_labels,  tokenizer, MAX_LENGTH)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint, num_labels=len(label_names), id2label=id2label, label2id=label2id,
    )
    args_kwargs = dict(
        output_dir=out_dir, learning_rate=learning_rate,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=epochs,
        weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        eval_strategy='epoch', logging_strategy='epoch',
        disable_tqdm=True, report_to='none',
        fp16=USE_FP16 and torch.cuda.is_available(), seed=seed,
    )
    if early_stop_patience is not None:
        args_kwargs.update(dict(
            save_strategy='epoch', save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model='f1_macro', greater_is_better=True,
        ))
    else:
        args_kwargs.update(dict(save_strategy='no'))

    args = TrainingArguments(**args_kwargs)
    trainer_cls = WeightedTrainer if use_class_weights else Trainer
    extra = dict(class_weights=class_weights_tensor) if use_class_weights else {}
    callbacks = []
    if early_stop_patience is not None:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=early_stop_patience))
    trainer = trainer_cls(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        processing_class=tokenizer, data_collator=data_collator,
        compute_metrics=compute_metrics, callbacks=callbacks, **extra,
    )
    try: trainer.remove_callback(NotebookProgressCallback)
    except Exception: pass
    print(f'\n=== {tag} | model={model_checkpoint} | lr={learning_rate} | weighted={use_class_weights} | ep={epochs} | seed={seed} ===')
    trainer.train()
    val_m  = trainer.evaluate(eval_dataset=val_ds)
    test_m = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix='test')
    del model, trainer
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return {
        'tag': tag, 'model': model_checkpoint,
        'lr': learning_rate, 'weighted': use_class_weights,
        'epochs': epochs, 'early_stop_patience': early_stop_patience, 'seed': seed,
        'val_f1_macro': val_m['eval_f1_macro'],
        'val_bal_acc': val_m['eval_balanced_accuracy'],
        'val_accuracy': val_m['eval_accuracy'],
        'test_f1_macro': test_m['test_f1_macro'],
        'test_bal_acc': test_m['test_balanced_accuracy'],
        'test_accuracy': test_m['test_accuracy'],
    }


## Part 1 — RoBERTa × 3 seeds on scrubbed+ text

In [6]:
roberta_seed_results = []
for s in SEEDS:
    r = run_one(
        model_checkpoint=ROBERTA_CKPT,
        learning_rate=ROBERTA_BEST_LR,
        use_class_weights=ROBERTA_BEST_WEIGHTED,
        seed=s, epochs=12, early_stop_patience=3,
        tag=f'roberta_scrubbed_plus_seed{s}',
    )
    roberta_seed_results.append(r)

roberta_df = pd.DataFrame(roberta_seed_results)[['seed','val_f1_macro','test_f1_macro','test_bal_acc','test_accuracy']]
print(roberta_df.round(4).to_string(index=False))
print()
print('Mean ± Std:')
print(roberta_df.drop(columns=['seed']).agg(['mean','std']).round(4).to_string())


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



=== roberta_scrubbed_plus_seed42 | model=roberta-base | lr=2e-05 | weighted=True | ep=12 | seed=42 ===
{'loss': '5.363', 'grad_norm': '25.49', 'learning_rate': '1.953e-05', 'epoch': '1'}
{'eval_loss': '2.407', 'eval_accuracy': '0.3646', 'eval_balanced_accuracy': '0.2327', 'eval_precision_macro': '0.2744', 'eval_recall_macro': '0.2327', 'eval_f1_macro': '0.1788', 'eval_runtime': '1.493', 'eval_samples_per_second': '685.4', 'eval_steps_per_second': '21.44', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '4.089', 'grad_norm': '22.47', 'learning_rate': '1.778e-05', 'epoch': '2'}
{'eval_loss': '1.724', 'eval_accuracy': '0.5249', 'eval_balanced_accuracy': '0.4801', 'eval_precision_macro': '0.4496', 'eval_recall_macro': '0.4801', 'eval_f1_macro': '0.4138', 'eval_runtime': '1.504', 'eval_samples_per_second': '680.3', 'eval_steps_per_second': '21.28', 'epoch': '2'}


...

{'train_runtime': '434.6', 'train_samples_per_second': '131.8', 'train_steps_per_second': '4.142', 'train_loss': '1.963', 'epoch': '12'}


There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

{'eval_loss': '1.54', 'eval_accuracy': '0.7283', 'eval_balanced_accuracy': '0.6249', 'eval_precision_macro': '0.6424', 'eval_recall_macro': '0.6249', 'eval_f1_macro': '0.6309', 'eval_runtime': '1.82', 'eval_samples_per_second': '562', 'eval_steps_per_second': '17.58', 'epoch': '12'}


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'test_loss': '1.338', 'test_accuracy': '0.7419', 'test_balanced_accuracy': '0.6501', 'test_precision_macro': '0.6592', 'test_recall_macro': '0.6501', 'test_f1_macro': '0.6505', 'test_runtime': '1.541', 'test_samples_per_second': '664', 'test_steps_per_second': '20.77', 'epoch': '12'}
 seed  val_f1_macro  test_f1_macro  test_bal_acc  test_accuracy
   42        0.6144         0.6386        0.6605         0.7283
  123        0.6215         0.6390        0.6556         0.7302
 2024        0.6309         0.6505        0.6501         0.7419

Mean ± Std:
      val_f1_macro  test_f1_macro  test_bal_acc  test_accuracy
mean        0.6223         0.6427        0.6554         0.7335
std         0.0083         0.0068        0.0052         0.0074


## Part 2 — ModernBERT × 3 seeds on scrubbed+ text

In [7]:
modernbert_seed_results = []
for s in SEEDS:
    r = run_one(
        model_checkpoint=MODERNBERT_CKPT,
        learning_rate=MODERNBERT_BEST_LR,
        use_class_weights=MODERNBERT_BEST_WEIGHTED,
        seed=s, epochs=12, early_stop_patience=3,
        tag=f'modernbert_scrubbed_plus_seed{s}',
    )
    modernbert_seed_results.append(r)

modernbert_df = pd.DataFrame(modernbert_seed_results)[['seed','val_f1_macro','test_f1_macro','test_bal_acc','test_accuracy']]
print(modernbert_df.round(4).to_string(index=False))
print()
print('Mean ± Std:')
print(modernbert_df.drop(columns=['seed']).agg(['mean','std']).round(4).to_string())

print()
print('--- Head-to-head on scrubbed+ ---')
print(f'RoBERTa    (weighted, lr=2e-5): {roberta_df["test_f1_macro"].mean():.4f} ± {roberta_df["test_f1_macro"].std():.4f}')
print(f'ModernBERT (plain,    lr=3e-5): {modernbert_df["test_f1_macro"].mean():.4f} ± {modernbert_df["test_f1_macro"].std():.4f}')


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.



=== modernbert_scrubbed_plus_seed42 | model=answerdotai/ModernBERT-base | lr=3e-05 | weighted=False | ep=12 | seed=42 ===
{'loss': '4.379', 'grad_norm': '20.47', 'learning_rate': '2.931e-05', 'epoch': '1'}
{'eval_loss': '1.805', 'eval_accuracy': '0.4594', 'eval_balanced_accuracy': '0.1848', 'eval_precision_macro': '0.1874', 'eval_recall_macro': '0.1848', 'eval_f1_macro': '0.157', 'eval_runtime': '11.71', 'eval_samples_per_second': '87.39', 'eval_steps_per_second': '2.734', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '2.938', 'grad_norm': '14.76', 'learning_rate': '2.668e-05', 'epoch': '2'}
{'eval_loss': '1.415', 'eval_accuracy': '0.5728', 'eval_balanced_accuracy': '0.313', 'eval_precision_macro': '0.5758', 'eval_recall_macro': '0.313', 'eval_f1_macro': '0.3232', 'eval_runtime': '11.87', 'eval_samples_per_second': '86.15', 'eval_steps_per_second': '2.695', 'epoch': '2'}


...

{'loss': '0.0002766', 'grad_norm': '0.03527', 'learning_rate': '1.241e-07', 'epoch': '12'}
{'eval_loss': '1.765', 'eval_accuracy': '0.6979', 'eval_balanced_accuracy': '0.541', 'eval_precision_macro': '0.6101', 'eval_recall_macro': '0.541', 'eval_f1_macro': '0.5658', 'eval_runtime': '11.01', 'eval_samples_per_second': '92.91', 'eval_steps_per_second': '2.906', 'epoch': '12'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '2574', 'train_samples_per_second': '22.25', 'train_steps_per_second': '0.699', 'train_loss': '0.978', 'epoch': '12'}
{'eval_loss': '1.761', 'eval_accuracy': '0.6989', 'eval_balanced_accuracy': '0.5435', 'eval_precision_macro': '0.6157', 'eval_recall_macro': '0.5435', 'eval_f1_macro': '0.5682', 'eval_runtime': '11.11', 'eval_samples_per_second': '92.07', 'eval_steps_per_second': '2.88', 'epoch': '12'}


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'test_loss': '1.6', 'test_accuracy': '0.7107', 'test_balanced_accuracy': '0.5593', 'test_precision_macro': '0.6203', 'test_recall_macro': '0.5593', 'test_f1_macro': '0.5762', 'test_runtime': '10.96', 'test_samples_per_second': '93.36', 'test_steps_per_second': '2.92', 'epoch': '12'}
 seed  val_f1_macro  test_f1_macro  test_bal_acc  test_accuracy
   42        0.5954         0.5939        0.5841         0.7331
  123        0.5742         0.6007        0.5803         0.7253
 2024        0.5682         0.5762        0.5593         0.7107

Mean ± Std:
      val_f1_macro  test_f1_macro  test_bal_acc  test_accuracy
mean        0.5793         0.5903        0.5746         0.7230
std         0.0143         0.0126        0.0134         0.0114

--- Head-to-head on scrubbed+ ---
RoBERTa    (weighted, lr=2e-5): 0.6427 ± 0.0068
ModernBERT (plain,    lr=3e-5): 0.5903 ± 0.0126


## Save results

In [8]:
out = {
    'notebook': '07.1_SeedHarness_FullText_Scrubbed_Plus',
    'text_column': TEXT_COLUMN,
    'text_columns_used': EXTRA_TEXT_COLS,
    'scrub_tiers': ['countries_and_adjectivals', 'coffee_region_aliases', 'cultivars', 'producer_context_terms'],
    'num_scrub_terms': len(all_scrub_terms),
    'post_scrub_leakage_rate': leak_rate,
    'roberta_seed_harness': {
        'model': ROBERTA_CKPT,
        'config': {'lr': ROBERTA_BEST_LR, 'weighted': ROBERTA_BEST_WEIGHTED, 'epochs': 12, 'early_stop_patience': 3},
        'seeds': SEEDS, 'runs': roberta_seed_results,
        'mean': roberta_df.drop(columns=['seed']).mean().to_dict(),
        'std':  roberta_df.drop(columns=['seed']).std().to_dict(),
    },
    'modernbert_seed_harness': {
        'model': MODERNBERT_CKPT,
        'config': {'lr': MODERNBERT_BEST_LR, 'weighted': MODERNBERT_BEST_WEIGHTED, 'epochs': 12, 'early_stop_patience': 3},
        'seeds': SEEDS, 'runs': modernbert_seed_results,
        'mean': modernbert_df.drop(columns=['seed']).mean().to_dict(),
        'std':  modernbert_df.drop(columns=['seed']).std().to_dict(),
    },
}
out_path = os.path.join(OUTPUT_DIR_ROOT, 'results.json')
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2, default=float)
print('Saved:', out_path)


Saved: artifacts/seed_harness_fulltext_scrubbed_plus\results.json
